<a href="https://colab.research.google.com/github/vidhya2026/Duffl_Career_Page/blob/main/RM_NCDL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Libraries

In [ ]:
!pip install sentence-transformers transformers pdfplumber python-docx spacy flask pyngrok -q
!python -m spacy download en_core_web_sm -q


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 37.0 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
#  Imports & Model Loading

import os, re, tempfile
import pdfplumber
import spacy
import pandas as pd
from docx import Document
from datetime import datetime
from sentence_transformers import SentenceTransformer, util
from transformers import pipeline
from flask import Flask, request, jsonify
from pyngrok import ngrok
from google.colab import files
from IPython.display import display, clear_output
import warnings
warnings.filterwarnings("ignore")

In [ ]:
print("Loading models...")
embedder   = SentenceTransformer('all-MiniLM-L6-v2')
ner_pipe   = pipeline("ner", model="dslim/bert-base-NER",    aggregation_strategy="simple")
ner_large  = pipeline("ner", model="dbmdz/bert-large-cased-finetuned-conll03-english", aggregation_strategy="simple")
nlp        = spacy.load("en_core_web_sm")
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")
print("Models loaded!")


Loading models...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Models loaded!


In [ ]:
# Text Extraction (PDF / DOCX / TXT)

def extract_text_pdf(path):
    full, left, right = "", "", ""
    with pdfplumber.open(path) as pdf:
        for page in pdf.pages:
            x0, y0, x1, y1 = page.bbox
            t = page.extract_text()
            if t: full += t + "\n"
            mid_x = x0 + (x1 - x0) * 0.42
            try:
                lt = page.crop((x0, y0, mid_x, y1)).extract_text()
                if lt: left += lt + "\n"
            except Exception: pass
            try:
                rt = page.crop((mid_x, y0, x1, y1)).extract_text()
                if rt: right += rt + "\n"
            except Exception: pass
    return full.strip(), left.strip(), right.strip()

def extract_text_docx(path):
    doc  = Document(path)
    text = "\n".join([p.text for p in doc.paragraphs if p.text.strip()])
    for table in doc.tables:
        for row in table.rows:
            for cell in row.cells:
                text += "\n" + cell.text
    return text.strip(), "", ""

def extract_text(path):
    if path.lower().endswith(".pdf"):  return extract_text_pdf(path)
    if path.lower().endswith(".docx"): return extract_text_docx(path)
    if path.lower().endswith(".txt"):
        with open(path, "r", errors="ignore") as f: c = f.read()
        return c, "", ""
    return "", "", ""

In [ ]:

# ════════════════════════════════════════════════════════════════════════
#  CELL 4 — Constants & Blocklists
# ════════════════════════════════════════════════════════════════════════

EDUCATION_KEYWORDS = [
    "university","college","institute","school","b.tech","m.tech",
    "b.e","m.e","bsc","msc","mba","bca","mca","bachelor","master",
    "degree","diploma","10th","12th","sslc","hsc","cbse","icse",
    "engineering college","polytechnic","iit","nit","bits","vit",
    "anna university","pune university","mumbai university",
    "graduation","post graduation","pg","ug","phd","doctorate",
    "higher secondary","secondary school","matriculation",
    "affiliated","board of",
]

# Patterns that strongly indicate an education institution line
# Used to block such lines from being treated as companies
EDUCATION_INSTITUTION_RE = re.compile(
    r'\b(university|college|institute of technology|iit|nit|bits|vit|'
    r'polytechnic|school of|academy|institution|deemed|'
    r'b\.?tech|m\.?tech|b\.?e\.?|m\.?e\.?|bsc|msc|mba|bca|mca|'
    r'higher secondary|matriculation|sslc|hsc|cbse|icse|'
    r'10th|12th|diploma|bachelor|master|phd|doctorate)\b',
    re.IGNORECASE
)

# Degree / year pattern that appears near education lines
# e.g. "2019 – 2023", "2018-2022" as college duration
EDUCATION_DATE_CONTEXT_RE = re.compile(
    r'\b(b\.?tech|m\.?tech|b\.?e|m\.?e|bsc|msc|mba|bca|mca|'
    r'bachelor|master|diploma|degree|10th|12th|sslc|hsc|phd)\b',
    re.IGNORECASE
)

TOOL_TECH_BLOCKLIST = {
    "balsamiq","figma","sketch","invision","zeplin","axure","adobe xd",
    "canva","miro","lucidchart","draw.io","whimsical","overflow",
    "framer","principle","marvel","proto.io","origami",
    "vscode","visual studio","eclipse","intellij","pycharm","sublime",
    "atom","notepad++","vim","emacs","webstorm","rider","xcode",
    "android studio","netbeans","codeblocks",
    "jira","confluence","trello","asana","notion","slack","teams",
    "basecamp","monday","clickup","github","gitlab","bitbucket",
    "jenkins","bamboo","circleci","travis",
    "react","angular","vue","django","flask","spring","laravel",
    "express","rails","fastapi","dotnet","asp.net","jquery",
    "bootstrap","tailwind","redux","graphql","rest","soap",
    "tensorflow","pytorch","keras","scikit","pandas","numpy",
    "matplotlib","seaborn","plotly","spark","hadoop","kafka",
    "mysql","postgresql","mongodb","sqlite","oracle","mssql",
    "redis","cassandra","dynamodb","firebase","supabase",
    "elasticsearch","neo4j","hbase",
    "aws","azure","gcp","docker","kubernetes","terraform",
    "ansible","puppet","chef","nginx","apache","linux","ubuntu",
    "centos","debian","windows server","vmware",
    "selenium","cypress","jest","mocha","junit","testng",
    "postman","soapui","appium","robotframework","cucumber",
    "loadrunner","jmeter","katalon",
    "tableau","power bi","qlik","looker","superset","metabase",
    "excel","google sheets","google analytics","mixpanel","segment",
    "agile","scrum","kanban","devops","ci/cd","microservices",
    "wordpress","shopify","salesforce","sap","oracle erp",
    "photoshop","illustrator","premiere","after effects",
    "unity","unreal","godot",
}

# ── CITY-ONLY known places (no country/state names) ──
KNOWN_CITIES = {
    "bangalore","bengaluru","mumbai","delhi","new delhi","hyderabad",
    "chennai","pune","kolkata","ahmedabad","surat","jaipur","lucknow",
    "noida","gurgaon","gurugram","chandigarh","bhopal","indore","nagpur",
    "vizag","visakhapatnam","kochi","cochin","coimbatore","madurai",
    "trichy","tiruchirappalli","trivandrum","thiruvananthapuram",
    "thrissur","kozhikode","calicut","salem","erode","tirunelveli",
    "vellore","mysore","mysuru","hubli","belgaum","mangalore","udupi",
    "bhubaneswar","patna","ranchi","raipur","guwahati",
    "dehradun","shimla","srinagar","jammu",
    "jodhpur","udaipur","ajmer","kota","bikaner",
    "varanasi","prayagraj","kanpur","agra","meerut",
    "ghaziabad","faridabad","amritsar","ludhiana","jalandhar",
    "vadodara","rajkot","gandhinagar","anand",
    "navi mumbai","thane","aurangabad","nashik","kolhapur","solapur",
    "pondicherry","puducherry","ooty",
    "remote","hybrid","onsite",
    "new york","los angeles","chicago","houston","san francisco",
    "seattle","denver","boston","atlanta","miami","dallas","austin",
    "london","manchester","birmingham","toronto","vancouver","montreal",
    "sydney","melbourne","brisbane","perth",
    "dubai","abu dhabi","doha","singapore","kuala lumpur",
    "tokyo","osaka","seoul","taipei","hong kong","shanghai","beijing",
    "berlin","munich","paris","amsterdam","zurich","rome","madrid",
    "johannesburg","cape town","nairobi",
}

INVALID_WORDS = {
    "tmp","temp",
    "naukri","indeed","linkedin","monster","shine","foundit",
    "glassdoor","hirist","instahyre","apna","freshersworld",
    "timesjobs","workindia","internshala","iimjobs",
    "ui","ux","uiux","uxui","dev","devops","qa","ba","pm",
    "frontend","backend","fullstack","full","stack","data",
    "cloud","ml","ai","hr","it","ios","web","sde","swe",
    "react","angular","vue","java","python","dotnet","node",
    "flutter","android","php","ruby","golang","rust",
    "aws","azure","gcp","docker","kubernetes",
    "resume","curriculum","vitae","profile","contact","summary",
    "objective","experience","education","skills","address","phone",
    "email","mobile","github","portfolio","work","professional",
    "personal","details","information","career","about","project",
    "references","developer","engineer","manager","analyst",
    "designer","consultant","specialist","lead","head","senior",
    "junior","intern","fresher","candidate","applicant",
    "updated","new","final","cv","doc","copy","sample","test",
    "example","demo","blank","template","format","latest","hire",
    "upload","document","file","draft","version",
}

SPECIFIC_TECH_DOMAIN_MAP = [
    (re.compile(
        r'\b(qa\s+engineer|quality\s+assurance\s+engineer|quality\s+analyst|'
        r'automation\s+tester|manual\s+tester|software\s+tester|sdet|'
        r'test\s+engineer|testing\s+engineer|qa\s+lead|qa\s+analyst|'
        r'associate\s+qa|junior\s+(?:software\s+)?tester|'
        r'software\s+test(?:ing)?\s+intern(?:ship)?|'
        r'qa\s+intern(?:ship)?|test(?:ing)?\s+intern(?:ship)?|'
        r'software\s+(?:quality|testing)|quality\s+engineer)\b',
        re.IGNORECASE), "QA / Test Engineer"),
    (re.compile(r'\b(\.net|asp\.net|c#|csharp|wpf|xaml|blazor|mvc|entity\s+framework|ef\s+core)\b', re.IGNORECASE), ".NET Developer"),
    (re.compile(r'\b(java\s+developer|spring\s+boot|spring\s+mvc|hibernate|j2ee|javaee)\b', re.IGNORECASE), "Java Developer"),
    (re.compile(r'\b(python\s+developer|django\s+developer|fastapi\s+developer|flask\s+developer)\b', re.IGNORECASE), "Python Developer"),
    (re.compile(r'\b(react\s+developer|react\.js\s+developer|next\.js\s+developer)\b', re.IGNORECASE), "React Developer"),
    (re.compile(r'\b(angular\s+developer|angularjs\s+developer)\b', re.IGNORECASE), "Angular Developer"),
    (re.compile(r'\b(node\s+developer|node\.js\s+developer|nodejs\s+developer)\b', re.IGNORECASE), "Node.js Developer"),
    (re.compile(r'\b(flutter\s+developer|dart\s+developer)\b', re.IGNORECASE), "Flutter Developer"),
    (re.compile(r'\b(android\s+developer|kotlin\s+developer)\b', re.IGNORECASE), "Android Developer"),
    (re.compile(r'\b(ios\s+developer|swift\s+developer)\b', re.IGNORECASE), "iOS Developer"),
    (re.compile(r'\b(data\s+scientist|machine\s+learning\s+engineer|ml\s+engineer|ai\s+engineer)\b', re.IGNORECASE), "Data Scientist / ML Engineer"),
    (re.compile(r'\b(data\s+analyst|bi\s+analyst|business\s+intelligence\s+analyst)\b', re.IGNORECASE), "Data Analyst"),
    (re.compile(r'\b(data\s+engineer|etl\s+developer)\b', re.IGNORECASE), "Data Engineer"),
    (re.compile(r'\b(devops\s+engineer|cloud\s+engineer|site\s+reliability\s+engineer|sre|infrastructure\s+engineer|platform\s+engineer)\b', re.IGNORECASE), "DevOps / Cloud Engineer"),
    (re.compile(r'\b(ui[\s/]*ux\s+designer|product\s+designer|ux\s+designer|ui\s+designer|ux\s+researcher|interaction\s+designer|visual\s+designer)\b', re.IGNORECASE), "UI/UX Designer"),
    (re.compile(r'\b(full[\s\-]?stack\s+developer|fullstack\s+developer)\b', re.IGNORECASE), "Full Stack Developer"),
    (re.compile(r'\b(front[\s\-]?end\s+developer|frontend\s+developer)\b', re.IGNORECASE), "Frontend Developer"),
    (re.compile(r'\b(back[\s\-]?end\s+developer|backend\s+developer)\b', re.IGNORECASE), "Backend Developer"),
    (re.compile(r'\b(product\s+manager|product\s+owner)\b', re.IGNORECASE), "Product Manager"),
    (re.compile(r'\b(project\s+manager|scrum\s+master|delivery\s+manager|program\s+manager)\b', re.IGNORECASE), "Project Manager"),
    (re.compile(r'\b(business\s+analyst|functional\s+analyst|system\s+analyst)\b', re.IGNORECASE), "Business Analyst"),
    (re.compile(r'\b(hr\s+(manager|executive)|talent\s+acquisition|recruiter)\b', re.IGNORECASE), "Human Resources"),
    (re.compile(r'\b(sales\s+(manager|executive)|digital\s+marketing\s+(executive|manager)|seo\s+analyst)\b', re.IGNORECASE), "Sales / Marketing"),
    (re.compile(r'\b(financial\s+analyst|accountant|finance\s+manager)\b', re.IGNORECASE), "Finance / Accounting"),
    (re.compile(r'\b(network\s+engineer|network\s+administrator|system\s+administrator)\b', re.IGNORECASE), "Network Engineer"),
    (re.compile(r'\b(security\s+engineer|cybersecurity\s+engineer|penetration\s+tester|information\s+security)\b', re.IGNORECASE), "Cybersecurity Engineer"),
    (re.compile(r'\b(technical\s+writer|content\s+writer)\b', re.IGNORECASE), "Technical Writer"),
    (re.compile(r'\b(software\s+engineer|software\s+developer|application\s+developer|sde|swe)\b', re.IGNORECASE), "Software Developer"),
]

DOMAIN_GROUP_MAP = {
    ".NET Developer"              : "Software Development",
    "Java Developer"              : "Software Development",
    "Python Developer"            : "Software Development",
    "React Developer"             : "Software Development",
    "Angular Developer"           : "Software Development",
    "Vue Developer"               : "Software Development",
    "Node.js Developer"           : "Software Development",
    "Full Stack Developer"        : "Software Development",
    "Frontend Developer"          : "Software Development",
    "Backend Developer"           : "Software Development",
    "Software Developer"          : "Software Development",
    "Flutter Developer"           : "Mobile Development",
    "Android Developer"           : "Mobile Development",
    "iOS Developer"               : "Mobile Development",
    "Data Scientist / ML Engineer": "Data Science / ML",
    "Data Analyst"                : "Data Science / ML",
    "Data Engineer"               : "Data Science / ML",
    "DevOps / Cloud Engineer"     : "DevOps / Cloud",
    "QA / Test Engineer"          : "QA / Testing",
    "UI/UX Designer"              : "UI/UX Design",
    "Product Manager"             : "Product Management",
    "Project Manager"             : "Project Management",
    "Business Analyst"            : "Business Analysis",
    "Human Resources"             : "Human Resources",
    "Sales / Marketing"           : "Sales / Marketing",
    "Finance / Accounting"        : "Finance / Accounting",
    "Network Engineer"            : "Network Engineering",
    "Cybersecurity Engineer"      : "Cybersecurity",
    "Technical Writer"            : "Technical Writing",
    "Unknown"                     : "Unknown",
}

DOMAIN_LABELS = [
    "Software Development", "QA / Testing", "Data Science / ML",
    "DevOps / Cloud", "UI/UX Design", "Mobile Development",
    "Product Management", "Business Analysis", "Project Management",
    "Human Resources", "Sales / Marketing", "Finance / Accounting",
    "Cybersecurity", "Network Engineering", "Technical Writing",
]

MODEL_CONFIDENCE_THRESHOLD = 0.55
SCORE_ZERO_THRESHOLD       = 35

# ── Section header patterns ──
NON_EXPERIENCE_SECTION_RE = re.compile(
    r'^\s*('
    r'projects?|project\s+details?|academic\s+projects?|personal\s+projects?|'
    r'key\s+projects?|major\s+projects?|notable\s+projects?|'
    r'skills?|technical\s+skills?|core\s+skills?|key\s+skills?|'
    r'certifications?|achievements?|awards?|publications?|'
    r'interests?|hobbies|extra.?curricular|activities|'
    r'education|academic|qualification|scholastic|'
    r'educational\s+background|academic\s+background|academic\s+details?|'
    r'educational\s+details?|academic\s+qualifications?'
    r')\s*$',
    re.IGNORECASE
)

EXPERIENCE_SECTION_RE = re.compile(
    r'^\s*('
    r'work\s+experience|professional\s+experience|employment\s+history|'
    r'experience|career\s+history|work\s+history|internship|'
    r'work\s+history\s+&\s+experience|employment\s+details?|'
    r'professional\s+background|career\s+summary'
    r')\s*$',
    re.IGNORECASE
)

# ── Education section detection (broader — catches inline edu sections) ──
EDUCATION_SECTION_RE = re.compile(
    r'^\s*('
    r'education|academic|qualification|scholastic|'
    r'educational\s+background|academic\s+background|'
    r'academic\s+details?|educational\s+details?|academic\s+qualifications?'
    r')\s*$',
    re.IGNORECASE
)

# ── Date range regex ──
_DATE_RANGE_RE = re.compile(
    r'(?:jan(?:uary)?|feb(?:ruary)?|mar(?:ch)?|apr(?:il)?|may|jun(?:e)?|'
    r'jul(?:y)?|aug(?:ust)?|sep(?:tember)?|oct(?:ober)?|nov(?:ember)?|dec(?:ember)?)'
    r'\s+\d{4}'
    r'|\d{4}\s*[-–]\s*(?:\d{4}|present|current|till\s*date)',
    re.IGNORECASE
)

# ── Role words ──
_ROLE_WORDS = re.compile(
    r'\b(engineer|developer|analyst|designer|manager|consultant|lead|head|'
    r'intern(?:ship)?|executive|specialist|architect|tester|scientist|associate|'
    r'coordinator|officer|director|programmer|technician|administrator|'
    r'devops|sde|swe|qa|ux|ui|frontend|backend|fullstack|trainee|'
    r'apprentice)\b',
    re.IGNORECASE
)

MONTH_MAP = {
    "jan":1,"feb":2,"mar":3,"apr":4,"may":5,"jun":6,
    "jul":7,"aug":8,"sep":9,"oct":10,"nov":11,"dec":12,
    "january":1,"february":2,"march":3,"april":4,"june":6,
    "july":7,"august":8,"september":9,"october":10,
    "november":11,"december":12
}
MONTH_NAMES = {
    1:"Jan",2:"Feb",3:"Mar",4:"Apr",5:"May",6:"Jun",
    7:"Jul",8:"Aug",9:"Sep",10:"Oct",11:"Nov",12:"Dec"
}

_TITLE_NAME_RE = re.compile(
    r'\b([A-Z][a-z]{1,20}(?:\s+[A-Z][a-z]{1,20}){1,3})\b'
)

FRESHER_RE = re.compile(
    r'\b(fresher|fresh\s+graduate|no\s+experience|0\s+year|zero\s+experience|'
    r'recent\s+graduate|entry[\s\-]level\s+fresher|just\s+graduated|newly\s+graduated|'
    r'campus\s+hire|campus\s+placement)\b',
    re.IGNORECASE
)

JOB_TITLE_LINE_RE = re.compile(
    r'(?:^|\n)\s*'
    r'((?:senior|junior|jr\.?|sr\.?|lead|principal|associate|staff|assistant|trainee)?\s*'
    r'(?:'
    r'(?:qa|quality\s+assurance|quality\s+analyst|test(?:ing)?)\s*(?:engineer|analyst|lead|intern(?:ship)?)?|'
    r'software\s+test(?:er|ing)?\s*(?:intern(?:ship)?)?|'
    r'automation\s+tester|manual\s+tester|sdet|'
    r'(?:\.net|asp\.net|c#|csharp)\s+developer|'
    r'software\s+(?:engineer|developer)|'
    r'(?:frontend|front[\s\-]end|backend|back[\s\-]end|full[\s\-]?stack)\s+developer|'
    r'(?:mobile|android|ios|flutter|react\s+native)\s+developer|'
    r'data\s+(?:scientist|analyst|engineer)|'
    r'(?:machine\s+learning|ml|ai)\s+engineer|'
    r'(?:devops|cloud|platform|infrastructure|systems?)\s+engineer|'
    r'site\s+reliability\s+engineer|sre|'
    r'(?:ui[\s/]*ux|product|graphic|visual|interaction)\s+designer|'
    r'product\s+(?:manager|owner)|'
    r'business\s+analyst|project\s+manager|scrum\s+master|'
    r'(?:hr|human\s+resource)\s+(?:manager|executive)|recruiter|'
    r'(?:sales|marketing)\s+(?:manager|executive)|'
    r'(?:network|security|cybersecurity)\s+engineer|'
    r'technical\s+writer|financial\s+analyst|accountant|'
    r'sde|swe'
    r'))',
    re.IGNORECASE | re.MULTILINE
)

# Metadata label pattern
_METADATA_LABEL_RE = re.compile(
    r'^(role|company|duration|location|responsibilities|period)\s*:\s*(.+)$',
    re.IGNORECASE
)

# "Company, City" pattern — e.g. "TCS, Bangalore" or "Infosys | Pune"
_COMPANY_CITY_INLINE_RE = re.compile(
    r'^([A-Z][A-Za-z\s&\.\-]{2,50}?)\s*[,|]\s*([A-Za-z\s]{3,30})$'
)


In [ ]:

# Core Line Classifiers


def _is_date_line(line):
    return bool(_DATE_RANGE_RE.search(line))

def _is_bullet_line(line):
    stripped = line.strip()
    if not stripped: return False
    if stripped[0] in ('-', '•', '*', '·', '–', '○', '◦', '▪', '▸'): return True
    # Long lines are descriptions, not headers — but don't treat as bullet
    # (we handle them separately in location scanning)
    return False

def _is_education_line(line):
    """
    Returns True if this line is clearly an education institution or degree line.
    Used to prevent edu lines from being treated as company/role lines.
    """
    return bool(EDUCATION_INSTITUTION_RE.search(line))

def _is_company_line_strict(line):
    """
    Strict company-line detection.
    - Short (3–80 chars), starts with capital
    - No date, no bullet, not an education institution line
    - Allows ONE comma (e.g. "TCS, Bangalore")
    - Not a metadata label line
    - Not a personal info line
    """
    line = line.strip()
    if not line or len(line) < 3 or len(line) > 80: return False
    if _is_date_line(line): return False
    if _is_bullet_line(line): return False
    if _is_education_line(line): return False          # ← KEY FIX
    if not re.match(r'^[A-Z]', line): return False
    if re.match(
        r'^(location|duration|role|company|responsibilities|period|'
        r'address|phone|email|mobile|linkedin|github|website|dob|'
        r'date\s+of\s+birth|nationality|objective|summary|skills?|'
        r'certif|language|declaration|reference)\s*:',
        line, re.IGNORECASE
    ): return False
    words = line.split()
    if len(words) > 10: return False
    # Allow one comma (city suffix), but not multiple
    if line.count(',') > 1: return False
    lowercase_count = sum(1 for w in words if w and w[0].islower())
    if lowercase_count > 2: return False
    return True

_is_company_line = _is_company_line_strict


def _parse_metadata_line(line):
    """
    If line is a metadata label (e.g. "Role: Tester", "Company: TCS"),
    return (label_type, value). Otherwise return (None, None).
    """
    m = _METADATA_LABEL_RE.match(line.strip())
    if m:
        return m.group(1).lower(), m.group(2).strip()
    return None, None


In [ ]:
# Section Splitter (FIXED)


def _find_education_start(lines):
    """
    Find the line index where education section starts.
    Checks:
    1. Explicit education section header
    2. Line containing degree keyword + year (inline edu start)
    3. Fallback: 60% of document
    Returns the split index.
    """
    n = len(lines)

    # Pass 1: explicit education header
    for i, line in enumerate(lines):
        if EDUCATION_SECTION_RE.match(line):
            return i


    start_search = int(n * 0.30)
    for i in range(start_search, n):
        s = lines[i].strip()
        if not s:
            continue
        # Line contains degree keyword
        if EDUCATION_DATE_CONTEXT_RE.search(s) and len(s) < 80:
            return i
        # Line contains institution keyword
        if EDUCATION_INSTITUTION_RE.search(s) and len(s) < 80:
            return i

    # Fallback: 60% of document
    return int(n * 0.60)


def split_experience_education(text):
    """
    Split text into (experience_part, education_part).
    Also strips project/skill sections from experience_part.
    Prevents education dates/institutions from leaking into experience_part.
    """
    lines = text.splitlines()
    edu_split = _find_education_start(lines)

    candidate_lines = lines[:edu_split]

    # Remove non-experience sections (projects, skills, etc.) from experience
    clean_exp_lines = []
    skipping = False
    for line in candidate_lines:
        if NON_EXPERIENCE_SECTION_RE.match(line):
            skipping = True
            continue
        if EXPERIENCE_SECTION_RE.match(line):
            skipping = False
            clean_exp_lines.append(line)
            continue
        if not skipping:
            clean_exp_lines.append(line)

    # Final safety: remove any remaining lines that are clearly education lines
    # (catches degree/institution lines that leaked through)
    final_exp_lines = []
    for line in clean_exp_lines:
        s = line.strip()
        # Skip lines that are clearly education-only (institution or degree, no role words)
        if (_is_education_line(s)
                and len(s) < 100
                and not _ROLE_WORDS.search(s)
                and not _is_date_line(s)):
            continue
        final_exp_lines.append(line)

    return "\n".join(final_exp_lines), "\n".join(lines[edu_split:])


In [ ]:
#Fresher Detection


def _is_edu_context_date(lines, date_idx, window=4):
    """
    Returns True if the date at lines[date_idx] is likely an education
    date (college year), not a work date.
    Checks: any line within `window` lines contains a degree keyword.
    """
    start = max(0, date_idx - window)
    end   = min(len(lines), date_idx + window + 1)
    for j in range(start, end):
        if j == date_idx:
            continue
        if EDUCATION_DATE_CONTEXT_RE.search(lines[j]):
            return True
        if EDUCATION_INSTITUTION_RE.search(lines[j]):
            return True
    return False


def _tag_lines(lines):
    """
    Tag each line as: 'date', 'role', 'company', 'edu', or 'other'.
    Education lines are tagged 'edu' and never used as anchors.
    """
    tags = []
    for i, line in enumerate(lines):
        s = line.strip()
        label_type, value = _parse_metadata_line(s)

        if _is_education_line(s):
            tags.append('edu')
        elif label_type == 'duration' or _is_date_line(s):
            # Check if this date is in an education context
            if _is_edu_context_date(lines, i):
                tags.append('edu')
            else:
                tags.append('date')
        elif label_type == 'role' and value:
            tags.append('role')
        elif label_type == 'company' and value:
            # Check if the company value looks like an education institution
            if _is_education_line(value):
                tags.append('edu')
            else:
                tags.append('company')
        elif (s
              and _ROLE_WORDS.search(s)
              and len(s) < 120
              and not _is_bullet_line(s)
              and not _is_education_line(s)
              and not re.match(
                  r'^(address|phone|email|mobile|linkedin|github|location|'
                  r'objective|summary|skills?|certif|dob|nationality)\s*:',
                  s, re.IGNORECASE)):
            tags.append('role')
        elif _is_company_line_strict(s):
            tags.append('company')
        else:
            tags.append('other')

    return tags


def _count_job_anchors_raw(experience_text):
    """
    Count distinct job/internship anchor positions in experience text.
    A job anchor = a date-range line (not an edu date) with a role or
    company line (not edu) within 8 lines above or 5 lines below it.
    """
    lines = experience_text.splitlines()
    n     = len(lines)
    tags  = _tag_lines(lines)

    anchor_positions = set()
    for i, tag in enumerate(tags):
        if tag != 'date':
            continue
        # Search window: 8 lines before, 5 lines after
        window_start = max(0, i - 8)
        window_end   = min(n, i + 6)
        for j in range(window_start, window_end):
            if j == i:
                continue
            if tags[j] in ('role', 'company'):
                anchor_positions.add(i)
                break

    return len(anchor_positions)


def is_fresher(full_text):
    """
    Returns True ONLY IF the candidate has no real work/internship experience.

    Logic:
    1. Split to get experience text (education/projects already removed).
    2. Tag lines — education lines tagged 'edu', never as anchors.
    3. Count job anchors (date + role/company pairs, edu-filtered).
    4. anchor_count >= 1 → Experienced.
    5. anchor_count == 0:
       - internship keyword + any non-edu date → Experienced
       - explicit fresher keyword → Fresher
       - otherwise → Fresher
    """
    experience_text, _ = split_experience_education(full_text)
    anchor_count = _count_job_anchors_raw(experience_text)

    if anchor_count >= 1:
        return False

    # Secondary: internship + non-edu date range anywhere in experience section
    internship_re = re.compile(r'\bintern(?:ship)?\b', re.IGNORECASE)
    if internship_re.search(experience_text):
        lines = experience_text.splitlines()
        for i, line in enumerate(lines):
            if _is_date_line(line) and not _is_edu_context_date(lines, i):
                return False   # internship + real date found → Experienced

    # Explicit fresher keyword
    if FRESHER_RE.search(full_text):
        return True

    # No anchors, no internship+date, no keyword → Fresher
    return True


def _extract_date_ranges_raw(text):
    pattern = re.compile(
        r'((?:jan(?:uary)?|feb(?:ruary)?|mar(?:ch)?|apr(?:il)?|may|jun(?:e)?|'
        r'jul(?:y)?|aug(?:ust)?|sep(?:tember)?|oct(?:ober)?|nov(?:ember)?|dec(?:ember)?)'
        r'\s+\d{4}|\d{4})'
        r'\s*(?:to|[-–])\s*'
        r'((?:jan(?:uary)?|feb(?:ruary)?|mar(?:ch)?|apr(?:il)?|may|jun(?:e)?|'
        r'jul(?:y)?|aug(?:ust)?|sep(?:tember)?|oct(?:ober)?|nov(?:ember)?|dec(?:ember)?)'
        r'\s+\d{4}|\d{4}|present|current|till\s*date)',
        re.IGNORECASE
    )
    return pattern.findall(text)

In [ ]:
#  Name Extraction
def get_name_from_filename(filepath):
    raw = os.path.splitext(os.path.basename(filepath))[0]
    raw = re.sub(r'\[.*?\]', ' ', raw)
    raw = re.sub(r'\(.*?\)', ' ', raw)
    raw = re.sub(r'([a-z])([A-Z])', r'\1 \2', raw)
    raw = re.sub(r'[_\-\.\s]+', ' ', raw).strip()
    tokens = raw.split()
    good = []
    for tok in tokens:
        tl = tok.lower()
        if not tok:                        continue
        if tl in INVALID_WORDS:            continue
        if any(c.isdigit() for c in tok):  continue
        if re.search(r'[^A-Za-z]', tok):   continue
        if len(tok) < 2:                   continue
        good.append(tok.capitalize())
    return " ".join(good) if good else None

def get_name_bert(text, model):
    try:
        entities = model(text[:700])
        names = [e['word'] for e in entities if e['entity_group'] == 'PER']
        if names:
            raw = " ".join(names[:4]).replace(" ##", "").strip()
            parts = [w for w in raw.split()
                     if w.lower() not in INVALID_WORDS
                     and not any(c.isdigit() for c in w)
                     and len(w) >= 2]
            if len(parts) >= 2:
                return " ".join(parts[:4])
    except: pass
    return None

def get_name_spacy(text):
    try:
        doc = nlp(text[:700])
        for ent in doc.ents:
            if ent.label_ == "PERSON":
                candidate = ent.text.strip()
                words = candidate.split()
                if (2 <= len(words) <= 5
                        and all(not any(c.isdigit() for c in w) for w in words)
                        and all(w.lower() not in INVALID_WORDS for w in words)):
                    return candidate
    except: pass
    return None

def get_name_regex(text):
    top = "\n".join(text.splitlines()[:12])
    for m in _TITLE_NAME_RE.findall(top):
        words = m.split()
        if all(w.lower() not in INVALID_WORDS for w in words):
            return m
    return None

def extract_name(full_text, left_col, right_col, filepath):
    fname_result = get_name_from_filename(filepath)
    if fname_result:
        return fname_result, "filename"
    top = "\n".join(full_text.splitlines()[:10])
    result = get_name_bert(top, ner_pipe)
    if result: return result, "bert-base-top"
    result = get_name_bert(top, ner_large)
    if result: return result, "bert-large-top"
    result = get_name_spacy(top)
    if result: return result, "spacy-top"
    result = get_name_bert(full_text, ner_pipe)
    if result: return result, "bert-base-full"
    result = get_name_bert(full_text, ner_large)
    if result: return result, "bert-large-full"
    result = get_name_spacy(full_text)
    if result: return result, "spacy-full"
    result = get_name_regex(full_text)
    if result: return result, "regex"
    return "Name Not Found", "none"

In [ ]:
# Career Break Detection

def parse_date(token):
    token = token.strip().lower()
    if token in ("present","current","till date","now"):
        now = datetime.now(); return now.month, now.year
    m = re.match(r'([a-z]+)\s+(\d{4})', token)
    if m:
        mon = MONTH_MAP.get(m.group(1)); yr = int(m.group(2))
        if mon: return mon, yr
    m = re.match(r'(\d{4})$', token)
    if m: return 6, int(m.group(1))
    return None

def extract_date_ranges(text):
    pattern = re.compile(
        r'((?:jan(?:uary)?|feb(?:ruary)?|mar(?:ch)?|apr(?:il)?|may|jun(?:e)?|'
        r'jul(?:y)?|aug(?:ust)?|sep(?:tember)?|oct(?:ober)?|nov(?:ember)?|dec(?:ember)?)'
        r'\s+\d{4}|\d{4})'
        r'\s*(?:to|[-–])\s*'
        r'((?:jan(?:uary)?|feb(?:ruary)?|mar(?:ch)?|apr(?:il)?|may|jun(?:e)?|'
        r'jul(?:y)?|aug(?:ust)?|sep(?:tember)?|oct(?:ober)?|nov(?:ember)?|dec(?:ember)?)'
        r'\s+\d{4}|\d{4}|present|current|till\s*date)',
        re.IGNORECASE
    )
    # Only use dates from experience section (not education)
    exp_text, _ = split_experience_education(text)
    ranges = []
    lines  = exp_text.splitlines()
    for match in pattern.finditer(exp_text):
        # Find line index of this match to check edu context
        char_pos  = match.start()
        line_idx  = exp_text[:char_pos].count('\n')
        if _is_edu_context_date(lines, line_idx):
            continue
        s = parse_date(match.group(1))
        e = parse_date(match.group(2))
        if s and e:
            start_m = s[1]*12 + s[0]; end_m = e[1]*12 + e[0]
            if end_m >= start_m:
                ranges.append((start_m, end_m, s, e))
    return sorted(ranges, key=lambda x: x[0])

def abs_to_monthyear(abs_m):
    yr = (abs_m-1)//12; mon = abs_m - yr*12; return mon, yr

def detect_career_break(text):
    ranges = extract_date_ranges(text)
    if len(ranges) < 2: return False, "No career break detected"
    breaks = []
    for i in range(1, len(ranges)):
        prev_end_abs   = ranges[i-1][1]
        curr_start_abs = ranges[i][0]
        gap            = curr_start_abs - prev_end_abs
        if gap >= 6:
            bsm, bsy = abs_to_monthyear(prev_end_abs)
            bem, bey = abs_to_monthyear(curr_start_abs)
            start_label = f"{MONTH_NAMES.get(bsm,'?')} {bsy}"
            end_label   = f"{MONTH_NAMES.get(bem,'?')} {bey}"
            yrs = gap//12; mons = gap%12
            if yrs > 0 and mons > 0: dur = f"{yrs}yr {mons}mo"
            elif yrs > 0:             dur = f"{yrs}yr{'s' if yrs>1 else ''}"
            else:                     dur = f"{mons}mo{'s' if mons>1 else ''}"
            breaks.append(f"{start_label} -> {end_label} ({dur})")
    if breaks:
        return True, "Career break: " + "  |  ".join(breaks)
    return False, "No career break detected"

In [ ]:
#  Domain Detection
def _extract_title_line_from_block(block_text):
    match = JOB_TITLE_LINE_RE.search(block_text)
    if match:
        return match.group(1).strip()
    return None

def _get_domain_from_title(title_line):
    if not title_line:
        return None
    for pattern, label in SPECIFIC_TECH_DOMAIN_MAP:
        if pattern.search(title_line):
            return label
    return None

def _get_domain_from_model(block_text):
    try:
        result = classifier(block_text[:600], DOMAIN_LABELS, multi_label=False)
        label, score = result["labels"][0], result["scores"][0]
        if score >= MODEL_CONFIDENCE_THRESHOLD:
            return label, score
    except:
        pass
    return None, 0.0

def resolve_domain_for_block(block_text):
    title = _extract_title_line_from_block(block_text)
    domain = _get_domain_from_title(title)
    if domain:
        return domain, f"title-regex ({title})"

    for line in block_text.splitlines()[:10]:
        stripped = line.strip()
        if not stripped or len(stripped) > 120:
            continue
        if _is_bullet_line(stripped):
            continue
        # Handle "Role: <title>" metadata
        label_type, value = _parse_metadata_line(stripped)
        if label_type == 'role' and value:
            domain = _get_domain_from_title(value)
            if domain:
                return domain, f"role-label ({value[:50]})"
            continue
        if re.match(
            r'^(location|duration|company|responsibilities|period)\s*:',
            stripped, re.IGNORECASE
        ):
            continue
        domain = _get_domain_from_title(stripped)
        if domain:
            return domain, f"line-scan ({stripped[:50]})"

    model_cat, confidence = _get_domain_from_model(block_text)
    if model_cat:
        return model_cat, f"model ({confidence:.0%})"

    return "Unknown", "no match"


def split_into_job_blocks(experience_text):
    """
    Split experience text into per-job blocks.
    Uses _tag_lines() so education lines are excluded from split logic.
    """
    lines = experience_text.splitlines()
    n     = len(lines)
    tags  = _tag_lines(lines)

    split_points = set()
    for i, tag in enumerate(tags):
        if tag != 'date':
            continue
        for j in range(max(0, i - 6), i):
            if tags[j] in ('company', 'role'):
                split_points.add(j)
                break

    split_points = sorted(split_points)

    if not split_points:
        # Fallback: split on date lines (non-edu)
        blocks, current = [], []
        for i, line in enumerate(lines):
            if tags[i] == 'date' and current:
                blocks.append("\n".join(current))
                current = [line]
            else:
                current.append(line)
        if current:
            blocks.append("\n".join(current))
        return [b for b in blocks if len(b.strip()) > 30]

    blocks, prev_idx = [], 0
    for sp in split_points:
        if sp > prev_idx:
            block = "\n".join(lines[prev_idx:sp])
            if len(block.strip()) > 30:
                blocks.append(block)
        prev_idx = sp
    last_block = "\n".join(lines[prev_idx:])
    if len(last_block.strip()) > 30:
        blocks.append(last_block)

    return blocks


def detect_domains_from_resume_text(full_text):
    fresher = is_fresher(full_text)
    experience_text, _ = split_experience_education(full_text)

    if fresher:
        title = _extract_title_line_from_block(full_text)
        domain = _get_domain_from_title(title)
        if not domain:
            domain, _ = _get_domain_from_model(full_text[:1200])
        return domain or "Unknown", None, False

    blocks = split_into_job_blocks(experience_text)

    if not blocks:
        domain, _ = resolve_domain_for_block(experience_text[:1500] or full_text[:1500])
        return domain or "Unknown", None, False

    block_domains = []
    for block in blocks:
        cat, _ = resolve_domain_for_block(block)
        if cat and cat != "Unknown":
            block_domains.append(cat)

    if not block_domains:
        return "Unknown", None, False

    deduped = []
    for cat in block_domains:
        if not deduped or deduped[-1] != cat:
            deduped.append(cat)

    current_domain = deduped[-1]
    current_group  = DOMAIN_GROUP_MAP.get(current_domain, current_domain)

    previous_domain = None
    for cat in deduped[:-1]:
        grp = DOMAIN_GROUP_MAP.get(cat, cat)
        if grp != current_group:
            previous_domain = cat

    if previous_domain:
        return current_domain, previous_domain, True

    return current_domain, None, False


In [ ]:
_LOCATION_LABEL_RE = re.compile(
    r'(?:^|\n)\s*location\s*:\s*([A-Za-z][A-Za-z\s\-\.]{1,40}?)(?:,|\n|$)',
    re.IGNORECASE | re.MULTILINE
)

def _extract_city_from_token(token):
    """
    Given a raw token string, try to find a KNOWN_CITIES match.
    Handles: "Bangalore", "Bangalore, India", "Bengaluru/Bangalore",
             "Navi Mumbai", "Remote / Hybrid"
    Returns the matched city string (title-cased) or None.
    """
    # Remove country/state suffix after comma
    token = token.split(',')[0].strip().lower()
    # Remove content after slash
    token = token.split('/')[0].strip()
    # Direct match
    if token in KNOWN_CITIES:
        return token.title()
    # Longest-prefix match
    for city in sorted(KNOWN_CITIES, key=len, reverse=True):
        if re.search(r'\b' + re.escape(city) + r'\b', token):
            return city.title()
    return None


def _get_header_lines_from_block(block):
    """
    Return all non-bullet, reasonably short lines from a block.
    Does NOT stop at first bullet — collects ALL header-area lines.
    Skips lines > 150 chars (descriptions).
    Collects up to 12 lines.
    """
    header_lines = []
    count = 0
    for line in block.splitlines():
        stripped = line.strip()
        if not stripped:
            continue
        if len(stripped) > 150:
            continue   # skip long description lines
        if _is_bullet_line(stripped):
            continue   # skip bullet lines
        header_lines.append(stripped)
        count += 1
        if count >= 12:
            break
    return header_lines


def extract_company_locations(full_text, left_col=None, right_col=None):
    """
    Extract ONLY work/office locations from job blocks in experience section.

    Per block (and also full experience text as fallback):
      1. Explicit "Location: <city>" metadata line
      2. "Company, City" inline pattern (e.g. "TCS, Bangalore")
      3. Keyword scan on header lines
      4. NER on header lines

    Never scans top-of-resume (home address area).
    """
    experience_text, _ = split_experience_education(full_text)
    if not experience_text.strip():
        return "Not mentioned"

    blocks = split_into_job_blocks(experience_text)

    # If block splitting failed, treat full experience as one block
    if not blocks:
        blocks = [experience_text]

    candidates = []

    for block in blocks:
        header_lines = _get_header_lines_from_block(block)
        header_text  = "\n".join(header_lines)

        # ── Method 1: Explicit "Location: <city>" label ──
        for m in _LOCATION_LABEL_RE.finditer(header_text + "\n" + block[:500]):
            raw = m.group(1).strip()
            city = _extract_city_from_token(raw)
            if city:
                candidates.append(city)

        # ── Method 2: "Company, City" or "Company | City" inline pattern ──
        for line in header_lines:
            # Skip if it's a metadata label line
            if re.match(
                r'^(location|duration|role|company|responsibilities|period)\s*:',
                line, re.IGNORECASE
            ):
                continue
            # Check for "Something, CityName" or "Something | CityName"
            m = re.match(r'^(.+?)\s*[,|]\s*([A-Za-z][A-Za-z\s]{2,25})$', line)
            if m:
                city = _extract_city_from_token(m.group(2).strip())
                if city:
                    candidates.append(city)
                    continue
            # Also try the whole line for city keywords
            city = _extract_city_from_token(line)
            if city:
                # Only accept if the line doesn't look like a bullet or long description
                candidates.append(city)

        # ── Method 3: NER on header lines ──
        if header_text.strip():
            try:
                for e in ner_pipe(header_text[:800]):
                    if e['entity_group'] in ('LOC', 'GPE'):
                        word = e['word'].replace(" ##", "").strip()
                        city = _extract_city_from_token(word)
                        if city:
                            candidates.append(city)
            except:
                pass

    # ── Deduplicate & validate against KNOWN_CITIES only ──
    seen, unique = set(), []
    for loc in candidates:
        loc_clean = loc.strip().title()
        loc_lower = loc_clean.lower()
        if len(loc_lower) < 2:
            continue
        if loc_lower in TOOL_TECH_BLOCKLIST:
            continue
        if any(c.isdigit() for c in loc_clean):
            continue
        if re.search(r'[^A-Za-z\s\.\-]', loc_clean):
            continue
        if any(kw in loc_lower for kw in EDUCATION_KEYWORDS):
            continue
        if loc_lower in KNOWN_CITIES and loc_lower not in seen:
            seen.add(loc_lower)
            unique.append(loc_clean)

    return ", ".join(unique) if unique else "Not mentioned"

In [ ]:
def count_companies(full_text, fresher=False):
    if fresher:
        return 0

    experience_text, _ = split_experience_education(full_text)
    if not experience_text.strip():
        return 0

    lines = experience_text.splitlines()
    n     = len(lines)

    # ── Strategy A: Labelled format ("Company: <name>") ──
    company_labels = []
    for line in lines:
        label_type, value = _parse_metadata_line(line)
        if label_type == 'company' and value:
            # Skip if the company value is an education institution
            if not _is_education_line(value):
                company_labels.append(value.lower().strip())

    if company_labels:
        return len(set(company_labels))

    # ── Strategy B: Structural detection with edu filtering ──
    tags = _tag_lines(lines)

    # Find all date anchors and their nearest company/role line index
    company_anchor_positions = []
    for i, tag in enumerate(tags):
        if tag != 'date':
            continue
        company_idx = None
        # Search 8 lines before
        for j in range(max(0, i - 8), i):
            if tags[j] in ('company', 'role'):
                company_idx = j
                break
        # Search 4 lines after if not found before
        if company_idx is None:
            for j in range(i + 1, min(n, i + 5)):
                if tags[j] in ('company', 'role'):
                    company_idx = j
                    break
        if company_idx is not None:
            company_anchor_positions.append(company_idx)

    # Deduplicate: if two anchor positions point to the same line index,
    # or are very close (within 4 lines) → same company (promotion scenario)
    if not company_anchor_positions:
        # Last fallback: count date clusters
        date_positions = [i for i, t in enumerate(tags) if t == 'date']
        if date_positions:
            clusters = 1
            for k in range(1, len(date_positions)):
                if date_positions[k] - date_positions[k-1] > 8:
                    clusters += 1
            return clusters
        return 0

    # Group anchor positions: positions within 5 lines = same company block
    company_anchor_positions = sorted(set(company_anchor_positions))
    groups = []
    current_group = [company_anchor_positions[0]]
    for idx in company_anchor_positions[1:]:
        if idx - current_group[-1] <= 5:
            current_group.append(idx)
        else:
            groups.append(current_group)
            current_group = [idx]
    groups.append(current_group)

    # Further deduplicate by company line text (handles same company multiple roles)
    unique_company_texts = set()
    for group in groups:
        # Use the company/role line text of the first anchor in the group
        anchor_line = lines[group[0]].strip().lower()
        # Normalise: remove city suffix (after comma)
        anchor_normalised = anchor_line.split(',')[0].strip()
        unique_company_texts.add(anchor_normalised)

    return len(unique_company_texts)


In [ ]:
def compute_match_score(jd_text, resume_text):
    jd_emb  = embedder.encode(jd_text,            convert_to_tensor=True)
    res_emb = embedder.encode(resume_text[:3000], convert_to_tensor=True)
    raw     = round(util.cos_sim(jd_emb, res_emb).item() * 100, 2)
    return 0.0 if raw < SCORE_ZERO_THRESHOLD else raw

In [ ]:
def build_summary(name, method, current_d, prev_d, switched,
                  has_break, break_desc, location_str, company_count,
                  fresher=False):
    lines = []
    lines.append(f"Candidate      : {name}  (via {method})")
    lines.append(f"Candidate Type : {'Fresher' if fresher else 'Experienced'}")
    lines.append(f"Current Domain : {current_d}")

    if fresher:
        lines.append("Domain Switch  : Not applicable (Fresher)")
        lines.append("Career Break   : Not applicable (Fresher)")
    else:
        if switched:
            lines.append(f"Domain Switch  : Yes  ({prev_d}  ->  {current_d})  [!]")
        else:
            lines.append("Domain Switch  : No")
        if has_break:
            lines.append(f"Career Break   : [!] {break_desc}")
        else:
            lines.append("Career Break   : No career break detected")

    lines.append(f"Work Locations : {location_str}")
    lines.append(
        f"Companies      : {company_count} company"
        if company_count == 1
        else f"Companies      : {company_count} companies"
    )
    return "\n".join(lines)

In [ ]:
# analyze_resume (main orchestrator)
def analyze_resume(jd_text, path, threshold=70, original_filename=None):
    name_path                      = original_filename if original_filename else path
    fname                          = os.path.basename(original_filename or path)
    full_text, left_col, right_col = extract_text(path)

    if not full_text.strip():
        return {
            "file"               : fname,
            "score"              : 0,
            "status"             : "ERROR",
            "name"               : "N/A",
            "candidate_type"     : "-",
            "domain"             : "-",
            "domain_switch"      : False,
            "domain_switch_label": "No",
            "domain_switch_from" : "",
            "career_break"       : False,
            "career_break_detail": "-",
            "locations"          : "Not mentioned",
            "company_count"      : 0,
            "summary"            : "Could not extract text from file."
        }

    name, method = extract_name(full_text, left_col, right_col, name_path)
    score        = compute_match_score(jd_text, full_text)
    status       = "MATCHED" if score >= threshold else "NOT MATCHED"

    fresher                     = is_fresher(full_text)
    current_d, prev_d, switched = detect_domains_from_resume_text(full_text)

    if fresher:
        has_break, break_desc = False, "Not applicable (Fresher)"
    else:
        has_break, break_desc = detect_career_break(full_text)

    location_str  = extract_company_locations(full_text)
    company_count = count_companies(full_text, fresher=fresher)

    switch_label = "Not applicable (Fresher)" if fresher else (
        f"Yes  ({prev_d}  ->  {current_d})" if switched else "No"
    )

    summary = build_summary(
        name, method, current_d, prev_d, switched,
        has_break, break_desc, location_str, company_count, fresher
    )

    return {
        "file"               : fname,
        "score"              : score,
        "status"             : status,
        "name"               : name,
        "candidate_type"     : "Fresher" if fresher else "Experienced",
        "domain"             : current_d,
        "domain_switch"      : switched if not fresher else False,
        "domain_switch_label": switch_label,
        "domain_switch_from" : prev_d if (switched and not fresher) else "",
        "career_break"       : has_break,
        "career_break_detail": break_desc,
        "locations"          : location_str,
        "company_count"      : company_count,
        "summary"            : summary
    }


In [ ]:
#  Display Results
def display_results(results, threshold):
    clear_output(wait=True)
    print(f"\n{'='*75}")
    print(f"   RESUME MATCHING REPORT  |  Threshold: {threshold}%")
    print(f"{'='*75}\n")

    matched = sum(1 for r in results if r.get("status") == "MATCHED")
    print(f"Total: {len(results)}  |  Matched: {matched}  |  Not Matched: {len(results)-matched}\n")
    print("-"*75)

    for r in sorted(results, key=lambda x: x.get("score", 0), reverse=True):
        print(f"\n  File        : {r['file']}")
        print(f"  Match Score : {r['score']}%  ->  {r['status']}")
        print(f"\n  -- Summary -----------------------------------------------")
        for line in r["summary"].splitlines():
            print(f"  {line}")
        print("-"*75)

    rows = []
    for r in results:
        name_line = next(
            (l for l in r["summary"].splitlines() if "Candidate" in l and "via" in l), ""
        )
        name_val = name_line.split(":", 1)[-1].split("(via")[0].strip() if name_line else "-"
        rows.append({
            "File"           : r["file"],
            "Name"           : name_val,
            "Type"           : r.get("candidate_type", "-"),
            "Score (%)"      : r["score"],
            "Status"         : r["status"],
            "Domain"         : r.get("domain", "-"),
            "Domain Switch"  : r.get("domain_switch_label", "No"),
            "Career Break"   : r.get("career_break_detail", "-"),
            "Work Locations" : r.get("locations", "Not mentioned"),
            "Companies"      : r.get("company_count", 0),
            "Summary"        : r["summary"].replace("\n", " | ")
        })

    df = pd.DataFrame(rows)
    print("\n SUMMARY TABLE:\n")
    display(df.sort_values("Score (%)", ascending=False).reset_index(drop=True))

In [ ]:
#   Flask API
app = Flask(__name__)

@app.route("/api/health", methods=["GET"])
def health():
    return jsonify({"status": "ok", "message": "Resume Matcher API is running"})


@app.route("/api/analyze-batch", methods=["POST"])
def analyze_batch():
    uploaded_files = request.files.getlist("files")
    jd_text_req    = request.form.get("jd_text", "")
    threshold      = int(request.form.get("threshold", 70))

    if not uploaded_files:
        return jsonify({"error": "No files uploaded"}), 400
    if not jd_text_req.strip():
        return jsonify({"error": "jd_text is required"}), 400

    results = []
    for resume_file in uploaded_files:
        raw_filename  = resume_file.filename or ""
        safe_filename = os.path.basename(raw_filename.replace("\\", "/")).strip()
        if not safe_filename: safe_filename = "unknown.pdf"
        suffix = os.path.splitext(safe_filename)[-1] or ".pdf"

        with tempfile.NamedTemporaryFile(delete=False, suffix=suffix) as tmp:
            resume_file.save(tmp.name)
            tmp_path = tmp.name
        try:
            result = analyze_resume(jd_text_req, tmp_path, threshold,
                                    original_filename=safe_filename)
            result["file"] = safe_filename
            results.append(result)
        except Exception as e:
            results.append({"file": safe_filename, "error": str(e)})
        finally:
            os.unlink(tmp_path)

    matched     = sum(1 for r in results if r.get("status") == "MATCHED")
    not_matched = len(results) - matched
    return jsonify({
        "total"      : len(results),
        "matched"    : matched,
        "not_matched": not_matched,
        "threshold"  : threshold,
        "results"    : sorted(results, key=lambda x: x.get("score", 0), reverse=True)
    })


@app.route("/api/analyze", methods=["POST"])
def analyze_single():
    if "file" not in request.files:
        return jsonify({"error": "No file uploaded"}), 400
    if not request.form.get("jd_text", "").strip():
        return jsonify({"error": "jd_text is required"}), 400

    resume_file = request.files["file"]
    jd_text_req = request.form["jd_text"]
    threshold   = int(request.form.get("threshold", 70))

    raw_filename  = resume_file.filename or ""
    safe_filename = os.path.basename(raw_filename.replace("\\", "/")).strip()
    if not safe_filename: safe_filename = "unknown.pdf"
    suffix = os.path.splitext(safe_filename)[-1] or ".pdf"

    with tempfile.NamedTemporaryFile(delete=False, suffix=suffix) as tmp:
        resume_file.save(tmp.name)
        tmp_path = tmp.name
    try:
        result = analyze_resume(jd_text_req, tmp_path, threshold,
                                original_filename=safe_filename)
        result["file"] = safe_filename
        return jsonify(result)
    except Exception as e:
        return jsonify({"error": str(e)}), 500
    finally:
        os.unlink(tmp_path)

In [25]:
# Colab Interactive Main

print("\n" + "="*75)
print("  RESUME MATCHER -- Match Score + Consolidated Summary")
print("="*75)

print("\n STEP 1: Paste Job Description. Type END when done:\n")
jd_lines = []
while True:
    line = input()
    if line.strip().upper() == "END":
        break
    jd_lines.append(line)
jd_text = "\n".join(jd_lines)
print(f" JD captured -- {len(jd_text)} chars")

t = input("\n STEP 2: Threshold % (Enter = 70): ").strip()
threshold = int(t) if t.isdigit() else 70
print(f" Threshold: {threshold}%")

print("\n STEP 3: Upload resumes (PDF / DOCX / TXT):")
uploaded = files.upload()

resume_paths = []
for fname, fdata in uploaded.items():
    p = f"/content/{fname}"
    with open(p, "wb") as f:
        f.write(fdata)
    resume_paths.append(p)

print(f"\n {len(resume_paths)} file(s) uploaded. Analysing...\n")

results = []
for i, path in enumerate(resume_paths):
    print(f"  [{i+1}/{len(resume_paths)}] {os.path.basename(path)}")
    results.append(analyze_resume(jd_text, path, threshold))

display_results(results, threshold)


   RESUME MATCHING REPORT  |  Threshold: 50%

Total: 1  |  Matched: 1  |  Not Matched: 0

---------------------------------------------------------------------------

  File        : Naukri_SIVARAMANV[4y_0m].pdf
  Match Score : 67.07%  ->  MATCHED

  -- Summary -----------------------------------------------
  Candidate      : Sivaramanv  (via filename)
  Candidate Type : Fresher
  Current Domain : QA / Test Engineer
  Domain Switch  : Not applicable (Fresher)
  Career Break   : Not applicable (Fresher)
  Work Locations : Not mentioned
  Companies      : 0 companies
---------------------------------------------------------------------------

 SUMMARY TABLE:



,File,Name,Type,Score (%),Status,Domain,Domain Switch,Career Break,Work Locations,Companies,Summary
0,Naukri_SIVARAMANV[4y_0m].pdf,Sivaramanv,Fresher,67.07,MATCHED,QA / Test Engineer,Not applicable (Fresher),Not applicable (Fresher),Not mentioned,0,Candidate : Sivaramanv (via filename) | ...
